# Live risk testing: pull all trades via Quantifi DSP
Read-only: lists every trade with HQL, then builds pricers and computes PV.

In [1]:
import pandas as pd
import os
import time

# Toolkit import so the pricer objects returned by DSP can be used
from Quantifi.Toolkit.Base import Dt

# Quantifi DSP REST client
from qservices.QuantifiApiClient import *

In [2]:
# Environment: switch between Prod and UAT here
ENV = 'Prod'   # 'Prod' or 'UAT'
host = {'Prod': 'hb-dc1pd-qfidsp', 'UAT': 'hb-dc1ut-qfidsp'}[ENV]
os.environ["Q_DATA_SERVICE_URL_PREFIX"] = f"https://{host}:47200"
os.environ["Q_MARKET_ENV_SERVICE_URL_PREFIX"] = f"https://{host}:47201"
os.environ["Q_LOCAL_INSTALL"] = "True"

requests.packages.urllib3.disable_warnings()   # unverified HTTPS cert

AS_OF = '2026-11-12'      # pricing date
CALC_ENV = 'Official'
MAX_TRADES = 100_000       # data_query silently truncates at this many rows (default is 500)

## 1. List all trades

In [3]:
# Only Abhishek Bhutra's trades. He appears in the DB under several Trader.Name values
# (AB = Abhishek, AC = Aman Chind is excluded).
TRADERS = ['abhutra', 'AB', 'Abhishek Bhutra']
trader_list = ', '.join(f"'{t}'" for t in TRADERS)

all_trades = Connect.data_query(f"FROM Trade b WHERE b.Trader.Name IN ({trader_list})",
                                max_results=MAX_TRADES, verify_https_cert=False)
print('Trades returned:', len(all_trades))
if len(all_trades) >= MAX_TRADES:
    print('WARNING: hit MAX_TRADES, results are probably truncated; raise it.')

trade_ids = [t['TradeId'] for t in all_trades]
print('Keys on a trade record:', sorted(all_trades[0].keys()) if all_trades else None)

Trades returned: 9712
Keys on a trade record: ['$type', 'Amount', 'Counterparty', 'Groups', 'InitialMargin', 'IsCleared', 'Notes', 'ObjectId', 'Payment', 'PaymentCcy', 'PaymentSettle', 'Product', 'Settle', 'Strategy', 'SubStrategy', 'Tags', 'Termination', 'TradeId', 'TradeStatus', 'TradeType', 'Traded', 'TradedLevel', 'TradedLevelType', 'Trader']


In [4]:
# Flat static view of the trades (top-level scalar fields only)
trades_static = pd.json_normalize(all_trades, max_level=1)


def flatten_tags(df, col, prefix):
    """Turn a column of [{'Name':..., 'Value':...}, ...] into one column per tag name."""
    def to_dict(tags):
        if not isinstance(tags, list):
            return {}
        return {f"{prefix}{t['Name']}": t.get('Value') for t in tags if isinstance(t, dict) and 'Name' in t}
    wide = pd.DataFrame([to_dict(t) for t in df[col]], index=df.index)
    return pd.concat([df.drop(columns=col), wide], axis=1)


for col, prefix in [('Tags', 'Tag.'), ('Product.Tags', 'Product.Tag.')]:
    if col in trades_static:
        trades_static = flatten_tags(trades_static, col, prefix)

# Most recent trades first (parse to datetime for sorting; unparseable/missing dates go last)
if 'Traded' in trades_static:
    trades_static = (trades_static
                     .sort_values('Traded', ascending=False, na_position='last',
                                  key=lambda s: pd.to_datetime(s, errors='coerce', utc=True))
                     .reset_index(drop=True))

# Show every column (no "..." in the middle) and don't truncate cell text
with pd.option_context('display.max_columns', None, 'display.max_colwidth', None,
                       'display.width', None):
    display(trades_static.head(5))
print(f'{trades_static.shape[1]} columns; trader column(s):',
      [c for c in trades_static.columns if 'trader' in c.lower()])
trades_static.to_csv('all_trades_static.csv', index=False)

,$type,ObjectId,TradeId,TradeStatus,TradeType,Traded,Settle,Termination,Amount,Payment,PaymentSettle,TradedLevel,Notes,IsCleared,InitialMargin,PaymentCcy,Groups,TradedLevelType.$type,TradedLevelType.Name,TradedLevelType.ObjectId,Strategy.$type,Strategy.Name,Strategy.ObjectId,SubStrategy.$type,SubStrategy.Name,SubStrategy.ObjectId,Trader.$type,Trader.Name,Trader.ObjectId,Counterparty.$type,Counterparty.Name,Counterparty.ObjectId,Product.$type,Product.Name,Product.ObjectId,BookingEntity.$type,BookingEntity.Name,BookingEntity.ObjectId,ClearingBroker.$type,ClearingBroker.Name,ClearingBroker.ObjectId,Product.Description,Product.Effective,Product.Maturity,Product.DaysToSettle,Product.Ccy,Product.Notional,Product.ClearingBroker,Product.Premium,Product.DayCount,Product.Frequency,Product.Calendar,Product.BDConvention,Product.AccruedOnDefault,Product.PayRecoveryAtMaturity,Product.FixedRecovery,Product.FixedRecoveryRate,Product.PremiumInAdvance,Product.Seniority,Product.ReferenceEntity,Product.ReferenceObligation,Product.SettlementMethod,Product.PhysicalCreditSettlement,Product.CashCreditSettlement,Product.CreditEvents,Product.Obligation,Product.DeliverableObligation,Product.CalcAgentInfo,Product.CdsType,Product.Resets,Product.CycleRule,Product.AmortizationSchedule,Product.CouponSchedule,Product.BonusPremium,Product.ReferenceObligationOption,Product.TransactionType,Product.ISDADefinition,TradeDocumentation.$type,TradeDocumentation.ObjectId,TradeDocumentation.Documentation,TradeDocumentation.CalcAgentAsSpecifiedInMaster,TradeDocumentation.DocumentationType,TradeDocumentation.MasterAgreementType,Product.ExternalId,Product.CreditEvent,Product.CdoType,Product.Underlyings,Product.SharedBasket,Product.Attachment,Product.Detachment,Product.Freq,Product.AmortizePremium,Product.FeeGuaranteed,ClearingHouse.$type,ClearingHouse.Name,ClearingHouse.ObjectId,LeadTrade.$type,LeadTrade.TradeId,LeadTrade.ObjectId,Product.Underlying,Product.Expiration,Product.Type,Product.Style,Product.Strike,Product.SettlementType,Product.TrancheType,Product.CDSIndex,Tag.DealCodeDescription,Tag.DealCodeId,Tag.SecurityId,Tag.StrategyId,Tag.UniqueId,Tag.AssociatedCreditIndex
0,StockOptionTrade,258394028621068288,StockOption_0784563,Done,New,2026-11-12,2026-11-12,NaN,40.0,98400.00,2026-11-12,24.600000,[],False,0.0,USD,"[{'$type': 'Group', 'FullPath': 'Group Structure\Hudson Bay Capital\AB', 'ObjectId': '148055837749805057'}]",QuoteType,Stock Option Price,56576470318841880,Strategy,Credit,113434415614404864,SubStrategy,30439-AB,113997365567886848,User,AB,281474976716288,LegalEntity,BOAS,28147497671220992,StockOption,SPX 11/20/2026 P7450 Index,87257242780751104,LegalEntity,HBMFLTD,28147497671086592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Equity vs Credit,30439,OPIN007603,1116276,1116276,NaN
1,CDXTrade,255860753830670099,CDX0024648,Done,New,2026-11-12,2026-11-13,NaN,-3200000.0,-68853.95,2026-11-13,0.004780,[],True,0.0,EUR,"[{'$type': 'Group', 'FullPath': 'Group Structure\Hudson Bay Capital\AB', 'ObjectId': '148055837749805057'}]",QuoteType,CDX Spread,56576470318841859,Strategy,Manual,113434415614404609,SubStrategy,Deal Code 31825,113997365567891712,User,abhutra,281474976715264,LegalEntity,BSEF,28147497671130368,CDX,ITRAXX-EUROPES44V1-5Y,85568392920403469,LegalEntity,HBMFLTD,28147497671086592,LegalEntity,HBMFLTD-UBICE-AB,28147497671086593,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,LegalEntity,IntercontinentalExchange Inc,28147497671072322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CDSE001587,183761,183761,NaN
2,CDSTrade,254453378947116817,CDS0024644,Done,New,2026-11-10,2026-11

113 columns; trader column(s): ['Trader.$type', 'Trader.Name', 'Trader.ObjectId']


## 2. Load pricers (cached) and compute PV
Pricers are cached per (calc env, as-of) in `pricer_cache/` as XML in Parquet. Only trades not yet cached are requested from DSP, so rerunning is cheap; new trades are appended.

In [ ]:
import pricer_cache as pc

REFRESH = False   # True = ignore today's cache and re-pull every trade (e.g. to pick up amended trades)

# Fetches from DSP only trades not already cached for (CALC_ENV, AS_OF); new rows are appended.
# A trade is cached only if it loads AND prices, so failures are retried on every run.
t0 = time.time()
pc.build_cache(AS_OF, CALC_ENV, ENV, trade_ids=trade_ids, refresh=REFRESH)

# Everything below reads from the cache, so it is identical whether the pricers were just fetched or not
pricers = pc.load_cached_pricers(AS_OF, CALC_ENV, trade_ids=trade_ids)
errors = {k: v for k, v in pc.load_cache_errors(AS_OF, CALC_ENV).items() if k in set(trade_ids)}
print(f'{len(pricers)} pricers, {len(errors)} errors in {time.time()-t0:.0f}s')
if errors:
    with pd.option_context('display.max_colwidth', None):
        display(pd.Series(errors, name='LoadError').value_counts().head(10))


In [6]:
# Diagnose load errors: full messages, grouped by trade type and trader
meta = pd.DataFrame([{'TradeId': t['TradeId'], 'TradeType': t['$type'],
                      'Trader': (t.get('Trader') or {}).get('Name'),
                      'Status': t.get('TradeStatus')} for t in all_trades]).set_index('TradeId')
meta['Loaded'] = meta.index.isin(pricers.keys())
meta['Error'] = pd.Series(errors)

print(meta.groupby('TradeType')['Loaded'].agg(total='size', loaded='sum').assign(
    failed=lambda d: d.total - d.loaded).to_string())
print()
print(meta.groupby('Trader')['Loaded'].agg(total='size', loaded='sum').to_string())
print()
# Full, untruncated error text (pandas cut it to "No Trad...")
for msg, n in meta['Error'].dropna().value_counts().head(10).items():
    print(f'{n:6d}  {msg}\n')

                  total  loaded  failed
TradeType                              
BondTrade            53      53       0
CDOTrade             92      87       5
CDSTrade            935       0     935
CDXOptionTrade     1484    1484       0
CDXTrade           5520       0    5520
CDXTrancheTrade    1537       0    1537
StockOptionTrade     24      23       1
StockTrade           67      67       0

                 total  loaded
Trader                        
AB                3697     847
Abhishek Bhutra    681      11
abhutra           5334     856

  5517  System.Runtime.Serialization.SerializationException: _indexFactor: unknown field in type Quantifi.Toolkit.Pricers.CDXPricer
   at Quantifi.Toolkit.Base.Serialization.SimpleXmlWrapper.ReadFields(IReadOnlyDictionary`2 fields, Type type, XmlReader reader, SimpleXmlSerializer settings)+MoveNext()
   at Quantifi.Toolkit.Base.Serialization.SimpleXmlWrapper.ReadValue(XmlReader reader, SimpleXmlSerializer settings, Object uninitializedObje

In [7]:
rows = []
for tid, p in pricers.items():
    row = {'TradeId': tid, 'Pricer': type(p).__name__}
    try:
        row['Product'] = str(p.Product)
    except Exception:
        pass
    try:
        row['PV'] = p.Pv()
    except Exception as e:
        row['PvError'] = str(e)
    rows.append(row)

risk_df = pd.DataFrame(rows).set_index('TradeId')
display(risk_df.style.format({'PV': '{:,.2f}'}))
risk_df.to_csv('all_trades_pv.csv')

if errors:
    display(pd.Series(errors, name='LoadError'))

,Pricer,Product,PV
TradeId,,,
Bond_0372461,BondPricer,OBL 2.4% Apr30,0.00
Bond_0372466,BondPricer,OBL 2.4% Apr30,0.00
Bond_0382145,BondPricer,OBL 2.4% Apr30,0.00
Bond0022202,BondPricer,OBL 2.4% Apr30,0.00
Bond0022234,BondPricer,OBL 2.4% Apr30,0.00
Bond0022240,BondPricer,ATOS 5% Dec30,0.00
Bond0022271,BondPricer,ATOS 5% Dec30,0.00
Bond0022272,BondPricer,OBL 2.5% Oct29,0.00
Bond0022273,BondPricer,OBL 2.5% Oct29,0.00


CDS0015787           System.Runtime.Serialization.SerializationExce...
CDS0015788           System.Runtime.Serialization.SerializationExce...
CDS0015789           System.Runtime.Serialization.SerializationExce...
CDS0015790           System.Runtime.Serialization.SerializationExce...
CDS0015791           System.Runtime.Serialization.SerializationExce...
                                           ...                        
CDXTranche0024603    System.Runtime.Serialization.SerializationExce...
CDXTranche0024626    System.Runtime.Serialization.SerializationExce...
CDXTranche0024627    System.Runtime.Serialization.SerializationExce...
CDXTranche0024631    System.Runtime.Serialization.SerializationExce...
CDXTranche0024632    System.Runtime.Serialization.SerializationExce...
Name: LoadError, Length: 7998, dtype: str